# graphArt_LM: the frozen torchvision holdout, scored once

17 real torchvision blocks and networks that nothing was tuned on (frozen 31 Oct). Three
fine-tuned 1.5B adapters, no training:
- **GPU 0, the final pipeline**: `mixed-v2` writes the graph; its node list goes to the
  `wirer`, which writes the edges.
- **GPU 1, the previous submission**: the adapter of 26 Oct.

About 1 hour, capped at 2. **Run it once.** If it fails, do not re-run: send the log.

**Before running**: *GPU T4 x2*, *Internet on*, and **three adapters attached under Input**
(*Add Input* -> your datasets), uploaded from the zips in `dist/`:
`qwen2.5-coder-1.5b-graphart-lora-mixed-v2`, `qwen2.5-coder-1.5b-graphart-lora-wirer` and
`qwen2.5-coder-1.5b-graphart-lora`. Check the Input panel lists all three. Then
*Save Version -> Save & Run All (Commit)*, and download `graphart_holdout.zip` from *Output*.

Generated by `scripts/make_kaggle_notebook.py --holdout`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the base all three adapters were tuned from.
# SPLIT        the 17 frozen holdout samples (data/generated/FROZEN.json). Scored ONCE.
# LANES        one list per GPU, run side by side; the steps of a lane run in order.
#              GPU 0, the final pipeline: mixed-v2 (zero_shot), then the wirer handed
#              mixed-v2's node lists. GPU 1: the adapter of the 26 Oct submission.
#              Reply caps 8000, as the protocol fixes (the longest reference is 7220).
# EVAL_ARGS    leave empty: the protocol scores all 17.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; SPLIT = "torchvision_holdout"; LANES = [[("qwen2.5-coder-1.5b-graphart-lora-mixed-v2", "pipeline.run_prompting", ["--prompt", "zero_shot", "--max-tokens", "8000"]), ("qwen2.5-coder-1.5b-graphart-lora-wirer", "pipeline.run_two_step", ["--brief", "--max-tokens", "8000", "--step1-from", "results/hf__qwen2.5-coder-1.5b-graphart-lora-mixed-v2__zero_shot__torchvision_holdout.jsonl"])], [("qwen2.5-coder-1.5b-graphart-lora", "pipeline.run_prompting", ["--prompt", "zero_shot", "--max-tokens", "8000"])]]; TIME_BUDGET_HOURS = 2.0; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "7895065662bd3214a1ed7d48cfe19ca0cf4ac029f63400288e6dda58a4f49f64"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Two GPUs, and the three uploaded adapters

In [ ]:
import threading, zipfile

n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
if n < len(LANES):
    raise SystemExit(f"{len(LANES)} lanes need {len(LANES)} GPUs, this session has {n} -- set GPU T4 x2")
INPUT = Path(os.environ.get("GRAPHART_INPUT", "/kaggle/input"))
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)
ADAPTERS = list(dict.fromkeys(name for lane in LANES for name, _, _ in lane))

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package():
    with zipfile.ZipFile(WORK / "graphart_holdout.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file():
                    z.write(p, p.relative_to(REPO))

# Each adapter by its exact folder name (os.walk with followlinks: Kaggle may mount inputs
# through links, which Path.rglob does not follow on Python 3.13).
found = {Path(d).name: Path(d) for d, _, files in os.walk(INPUT, followlinks=True)
         if "adapter_config.json" in files and "adapter_model.safetensors" in files} if INPUT.exists() else {}
missing = [name for name in ADAPTERS if name not in found]
if missing:
    print(f"adapters found: {sorted(found)}")
    for d, dirs, files in os.walk(INPUT, followlinks=True):
        if d.count(os.sep) - str(INPUT).count(os.sep) <= 3:
            print(f"  {d}: {sorted(dirs)[:10]} {sorted(files)[:10]}")
    raise SystemExit(f"missing adapter(s) {missing} under /kaggle/input -- attach them as inputs (see the top). "
                     "Nothing has been scored, so it is safe to fix and run again.")
for name in ADAPTERS:
    cfg = json.loads((found[name] / "adapter_config.json").read_text())
    if cfg.get("base_model_name_or_path") != BASE_MODEL:
        raise SystemExit(f"{name} was tuned from {cfg.get('base_model_name_or_path')}, not {BASE_MODEL}")
    shutil.copytree(found[name], REPO / "lora" / name, dirs_exist_ok=True)
    print(f"using {name}", flush=True)

## 5. Score the holdout: one lane per GPU

In [ ]:
ENV = dict(os.environ, HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
codes = {}

def run_lane(gpu, lane):
    for name, runner, flags in lane:
        flags = [str(REPO / f) if f.startswith("results/") else f for f in flags]
        given = [f for f in flags if f.endswith(".jsonl")]
        if given and not Path(given[0]).exists():
            codes[name] = "skipped: no node lists to hand it"
            continue
        if remaining() < 300:
            codes[name] = "skipped: out of time"
            continue
        cmd = [sys.executable, "-m", runner, "--backend", "hf", "--model", str(REPO / "lora" / name),
               "--split", SPLIT, *flags, *EVAL_ARGS]
        print(f"##### {name} on GPU {gpu}: {' '.join(cmd[2:])}", flush=True)
        with open(LOGS / f"{name}.log", "w") as log:
            proc = subprocess.Popen(cmd, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)),
                                    stdout=log, stderr=subprocess.STDOUT)
            try:
                codes[name] = proc.wait(timeout=max(remaining() - 180, 60))
            except subprocess.TimeoutExpired:
                proc.kill()
                codes[name] = "stopped by the time budget"
        print(f"##### {name}: {codes[name]}", flush=True)

threads = [threading.Thread(target=run_lane, args=(gpu, lane)) for gpu, lane in enumerate(LANES)]
for t in threads:
    t.start()
for t in threads:
    t.join()
for name in ADAPTERS:
    text = (LOGS / f"{name}.log").read_text() if (LOGS / f"{name}.log").exists() else ""
    print(f"\n##### {name} ({codes.get(name)}), last lines:\n" + "\n".join(text.splitlines()[-8:]), flush=True)
package()
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:80s} n={m['n']} failed={m['failed']}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download graphart_holdout.zip")